#Read csv file using data frame reader API

In [0]:
%run ../00-common/01.environment-config



In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
source_path=f"{landing_folder_path}/results"
table_name=f"{catalog_name}.{bronze_schema}.results"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


result_schema = StructType([
  StructField('date', DateType(), True),
  StructField('raceName', StringType(), True),
  StructField('round', IntegerType(), True),
  StructField('season', IntegerType(), True),
  StructField('url', StringType(), True),
  StructField('constructorId', StringType(), True),
  StructField('driverId', StringType(), True),
  StructField('grid', IntegerType(), True),
  StructField('laps', IntegerType(), True),
  StructField('number', IntegerType(), True),
  StructField('points', DoubleType(), True),
  StructField('position', IntegerType(), True),
  StructField('positionText', StringType(), True),
  StructField('status', StringType(), True)
])
#spark.createDataFrame([], driver_schema).printSchema()
df_results = (spark.read.format("json")
               .option('header', True)
               .option('mode', 'FAILFAST')  # strict mode datatype validation
             #  .option('mod', 'PERMISSIVE') # ignore bad records with null value
              # .option('inferSchema', True)  optional incase of schema passing as below
               .schema(result_schema)
               .load(source_path))

In [0]:
df_results.show();

In [0]:
import pyspark.sql.functions as F

df_results_final=add_ingestion_metadata(df_results)
display(df_results_final)

In [0]:
(df_results_final.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(table_name))

In [0]:
%sql
select * from formula1_catalog.bronze.results;
--select count(*) from formula1_catalog.bronze.results

In [0]:
df_table=spark.read.table(table_name)
display(df_table)

In [0]:
%sql
select season, count(*) from formula1_catalog.bronze.results group by season order by season desc